# Analisis IPM Indonesia 2026

Notebook ringkas pendamping `scripts/analysis.py`. Semua angka dihitung dari CSV di `data/` (sumber: BPS, lihat `data/SOURCES.md`). Hasil lengkap ada di `findings.json` dan `charts/`.

Pasangan korelasi lintas proyek dinyatakan eksplisit: IPM 2025 x kemiskinan Mar 2025.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

nas = pd.read_csv('data/ipm_nasional.csv')
prov = pd.read_csv('data/ipm_provinsi.csv')
komp = pd.read_csv('data/komponen_provinsi_2025.csv')
print(nas.shape, prov.shape, komp.shape)
nas


## Tren nasional


In [ ]:
gain = nas.loc[nas.tahun==2025,'ipm'].iloc[0] - nas.loc[nas.tahun==2020,'ipm'].iloc[0]
print('IPM 2020:', nas.loc[nas.tahun==2020,'ipm'].iloc[0])
print('IPM 2025:', nas.loc[nas.tahun==2025,'ipm'].iloc[0])
print('Gain total poin:', round(gain,2), '| rata-rata per tahun:', round(gain/5,3))
nas[['tahun','uhh_tahun','hls_tahun','rls_tahun','pengeluaran_ribu_rp']]


In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
ax.plot(nas.tahun, nas.ipm, color='#2F5D8A', lw=2.8, marker='o')
ax.set_title('Chart 1 - Tren IPM Nasional 2020-2025')
ax.set_ylabel('IPM'); ax.grid(color='#EFE7DB')
plt.show()


## Ranking provinsi 2025


In [ ]:
p25 = prov[prov.tahun==2025].sort_values('ipm', ascending=False)
print('Tertinggi:', p25.iloc[0].provinsi, p25.iloc[0].ipm)
print('Terendah:', p25.iloc[-1].provinsi, p25.iloc[-1].ipm)
print('Gap poin:', round(p25.ipm.max()-p25.ipm.min(),2))
print(p25.kategori.value_counts().to_dict())
p25.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
d = p25.sort_values('ipm')
ax.barh(d.provinsi, d.ipm, color='#7FB0D9')
ax.set_title('Chart 2 - Ranking IPM Provinsi 2025')
ax.set_xlabel('IPM 2025'); ax.grid(color='#EFE7DB')
plt.show()


## Konvergensi 2022 ke 2025


In [ ]:
piv = prov.pivot(index='provinsi', columns='tahun', values='ipm')
piv['gain'] = piv[2025]-piv[2022]
print('Tercepat:')
print(piv.sort_values('gain', ascending=False).head(3))
print('Terlambat:')
print(piv.sort_values('gain').head(3))
print('Gain rata-rata 10 terbawah 2022:', round(piv.nsmallest(10,2022).gain.mean(),2))
print('Gain rata-rata 10 teratas 2022:', round(piv.nlargest(10,2022).gain.mean(),2))


## Komponen 2025: 10 provinsi terendah


In [ ]:
bawah = komp.nsmallest(10,'ipm')
print('Rata-rata 10 terendah:')
print(bawah[['uhh_tahun','hls_tahun','rls_tahun','pengeluaran_ribu_rp']].mean().round(2))
print('Gap HLS-RLS bawah:', round(bawah.hls_tahun.mean()-bawah.rls_tahun.mean(),2))
print('Gap HLS-RLS nasional:', round(13.30-9.07,2))
bawah[['provinsi','ipm','uhh_tahun','hls_tahun','rls_tahun','pengeluaran_ribu_rp']]


## Korelasi IPM 2025 x kemiskinan Mar 2025


In [ ]:
kem = pd.read_csv('/home/hatch/workspace/kemiskinan/data/kemiskinan_provinsi.csv')
kem25 = kem[kem.periode=='Mar 2025'][['provinsi','persen_miskin']]
g = komp.merge(kem25, on='provinsi', how='inner')
print('n provinsi:', len(g))
print('Pearson r:', round(g.ipm.corr(g.persen_miskin),4))
g.sort_values('ipm', ascending=False).head()


In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
ax.scatter(g.ipm, g.persen_miskin, color='#2F5D8A')
coef = np.polyfit(g.ipm, g.persen_miskin, 1)
xs = np.linspace(g.ipm.min(), g.ipm.max(), 10)
ax.plot(xs, np.polyval(coef, xs), color='#D96C8A', ls='--')
ax.set_title('Chart 3 - IPM 2025 vs Kemiskinan Mar 2025')
ax.set_xlabel('IPM 2025'); ax.set_ylabel('Penduduk miskin (%)'); ax.grid(color='#EFE7DB')
plt.show()


## Kesimpulan singkat

- IPM nasional naik dari 72,81 (2020) ke 75,90 (2025), tetap kategori Tinggi.
- Gap provinsi 2025 masih lebar: DKI Jakarta 85,05 vs Papua Pegunungan 54,91.
- Kelompok 10 terbawah 2022 tumbuh sedikit lebih cepat dari 10 teratas, tetapi gap maksimum-minimum hampir tidak berubah.
- Korelasi IPM dengan kemiskinan kuat dan negatif. Angka persisnya ada di `findings.json`.
